# From notes to memory: the ingestion pipeline

The core verbs take one fact or one edge at a time. `anatid.ingest` takes prose. A note goes to an
extractor, which proposes a *memory patch*: facts to add, facts to correct, relations to open and
close, names that mean an existing entity. The pipeline resolves the proposal against the graph,
shows it to a reviewer as a diff, and applies it in one transaction with the note stored first as
evidence.

This notebook runs the whole flow offline with `ScriptedExtractor`, which returns prepared patches
instead of calling a model, so every step is visible. The last cell shows how to plug in a real
model. It is the same story as [`examples/ingest_notes.py`](../ingest_notes.py), step by step.

In [1]:
from datetime import datetime

from anatid import Anatid, HashEmbedder
from anatid.ingest import (
    AddFact, Alias, Correction, MemoryPatch, Relation, ScriptedExtractor, Span,
    ingest, prepare, propose,
)

db = Anatid.open(":memory:", tenant=1, embedding_dim=64, embedder=HashEmbedder(dim=64))

NOTES = [
    (datetime(2026, 3, 2, 9), "notes/2026-03-02.md",
     "Ada leads the Kestrel team. Kestrel owns the ingest service, and Bo maintains it day to day."),
    (datetime(2026, 6, 15, 9), "notes/2026-06-15.md",
     "Bo moved to the platform group. Cy took over the ingest service from Bo this week."),
    (datetime(2026, 8, 20, 9), "notes/2026-08-20.md",
     "The ingest service must stay on Python 3.10 until the Kestrel team finishes the migration. "
     "Ada leads Kestrel."),
]
for when, source, text in NOTES:
    print(f"{when.date()}  {source}\n    > {text}")

2026-03-02  notes/2026-03-02.md
    > Ada leads the Kestrel team. Kestrel owns the ingest service, and Bo maintains it day to day.
2026-06-15  notes/2026-06-15.md
    > Bo moved to the platform group. Cy took over the ingest service from Bo this week.
2026-08-20  notes/2026-08-20.md
    > The ingest service must stay on Python 3.10 until the Kestrel team finishes the migration. Ada leads Kestrel.


## 1. A memory patch

A `MemoryPatch` is what an extractor returns. Every operation can carry a `Span`, the characters
of the note that support it, so a reviewer can check the evidence. `describe()` renders the patch
as the diff a reviewer sees; `to_json()` is the same patch on the wire.

In [2]:
n1 = NOTES[0][2]
patch1 = MemoryPatch(
    add_facts=(
        AddFact("Ada leads Kestrel", ("Ada", "Kestrel"), span=Span.locate("Ada leads the Kestrel team", n1)),
        AddFact("Kestrel owns the ingest service", ("Kestrel", "ingest service"),
                span=Span.locate("Kestrel owns the ingest service", n1)),
        AddFact("Bo maintains the ingest service", ("Bo", "ingest service"),
                span=Span.locate("Bo maintains it day to day", n1)),
    ),
    add_relations=(
        Relation("Ada", "Kestrel", "leads"),
        Relation("Kestrel", "ingest service", "owns"),
        Relation("Bo", "ingest service", "maintains"),
    ),
)
print(patch1.describe())
print()
print(patch1.to_json()[:300] + " ...")

memory patch: 3 facts, 3 relation(s) added
  + fact        "Ada leads Kestrel"  about: Ada, Kestrel
  + fact        "Kestrel owns the ingest service"  about: Kestrel, ingest service
  + fact        "Bo maintains the ingest service"  about: Bo, ingest service
  + relation    Ada -leads-> Kestrel
  + relation    Kestrel -owns-> ingest service
  + relation    Bo -maintains-> ingest service

{
  "source_text": "",
  "add_facts": [
    {
      "content": "Ada leads Kestrel",
      "entities": [
        "Ada",
        "Kestrel"
      ],
      "kind": "fact",
      "confidence": 1.0,
      "span": {
        "start": 0,
        "end": 26
      }
    },
    {
      "content": "Kestrel owns t ...


## 2. Propose, review, apply

`propose(db, text, extractor=...)` runs three steps and writes nothing: it collects the current
facts about the entities the note names (the extractor's context), asks the extractor for a patch,
and *prepares* the patch against the graph. `ingest(...)` adds the review hook and the apply. The
hook receives the prepared patch and returns the patch to apply, edited or not, or `None` to
decline.

In [3]:
def review(patch):
    print("proposed:")
    print("    " + patch.describe().replace("\n", "\n    "))
    return patch          # edit it with patch.replace(...), or return None to decline

extractor = ScriptedExtractor([patch1])
when, source, text = NOTES[0]
receipt = ingest(db, text, extractor=extractor, writer="notes-bot", source=source, review=review, now=when)
print("\napplied:", receipt.describe())

proposed:
    memory patch: 3 facts, 3 relation(s) added
    source: "Ada leads the Kestrel team. Kestrel owns the ingest service, and Bo maintains it day to..." (92 chars)
      + fact        "Ada leads Kestrel"  about: Ada, Kestrel  [0:26] "Ada leads the Kestrel team"
      + fact        "Kestrel owns the ingest service"  about: Kestrel, ingest service  [28:59] "Kestrel owns the ingest service"
      + fact        "Bo maintains the ingest service"  about: Bo, ingest service  [65:91] "Bo maintains it day to day"
      + relation    Ada -leads-> Kestrel
      + relation    Kestrel -owns-> ingest service
      + relation    Bo -maintains-> ingest service

applied: episode 884810224898391040 stored; 3 memories created (884810224906779648, 884810224940334080, 884810224969694208); 3 relations opened.


## 3. A correction, and the edge that follows it

The second note changes who maintains the service. The extractor proposes a *correction* that names
the old fact by its text; `prepare` resolves that to the memory's id, and because the correction
swaps exactly one entity (Bo for Cy) it also moves the `maintains` edge the old fact's note opened.
The extractor did not have to say so; the patch's notes show what the pipeline inferred.

In [4]:
n2 = NOTES[1][2]
patch2 = MemoryPatch(
    add_facts=(AddFact("Bo works in the platform group", ("Bo", "platform group"),
                       span=Span.locate("Bo moved to the platform group", n2)),),
    corrections=(Correction("Cy maintains the ingest service", old_text="Bo maintains the ingest service",
                            entities=("Cy", "ingest service"),
                            span=Span.locate("Cy took over the ingest service from Bo", n2)),),
    add_relations=(Relation("Bo", "platform group", "member_of"),),
)
prepared = prepare(patch2, db)
print(prepared.describe())
print("\npipeline notes:")
for line in prepared.notes:
    print("   ", line)

memory patch: 1 fact, 1 correction, 2 relation(s) added, 1 relation(s) removed
  + fact        "Bo works in the platform group"  about: Bo, platform group
  ~ correction  memory 884810224969694208 "Bo maintains the ingest service"
                -> "Cy maintains the ingest service"  about: Cy, ingest service
  - relation    Bo -maintains-> ingest service
  + relation    Bo -member_of-> platform group
  + relation    Cy -maintains-> ingest service
  note          handover: closed and opened maintains between 'Bo' and 'ingest service' for 'Cy', from the correction of memory 884810224969694208

pipeline notes:
    handover: closed and opened maintains between 'Bo' and 'ingest service' for 'Cy', from the correction of memory 884810224969694208


In [5]:
when, source, text = NOTES[1]
receipt = ingest(db, text, extractor=ScriptedExtractor([patch2]), writer="notes-bot", source=source, now=when)
print("applied:", receipt.describe())

applied: episode 884810225292655616 stored; 2 memories created (884810225296849920, 884810225326210048); 1 superseded (884810224969694208); 2 relations opened; 1 relations closed.


## 4. Aliases and duplicates

The third note calls Kestrel "the Kestrel team" and repeats something the graph already holds. An
`Alias` maps the note's name onto the existing entity, and `prepare` drops the restated fact with a
note saying which memory already holds it.

In [6]:
n3 = NOTES[2][2]
patch3 = MemoryPatch(
    add_facts=(
        AddFact("The ingest service must stay on Python 3.10 until the Kestrel migration finishes",
                ("ingest service", "Kestrel"), kind="constraint",
                span=Span.locate("The ingest service must stay on Python 3.10", n3)),
        AddFact("Ada leads Kestrel", ("Ada", "the Kestrel team")),
    ),
    entity_aliases=(Alias("the Kestrel team", "Kestrel", span=Span.locate("the Kestrel team", n3)),),
)
when, source, text = NOTES[2]
receipt = ingest(db, text, extractor=ScriptedExtractor([patch3]), writer="notes-bot", source=source, now=when)
print("applied:", receipt.describe())
for line in receipt.patch.notes:
    print("   ", line)

applied: episode 884810225590451200 stored; 1 memories created (884810225598839808); aliases 'the Kestrel team' -> 'Kestrel'.
    alias 'the Kestrel team' -> 'Kestrel': 1 reference rewritten
    dedupe: dropped 'Ada leads Kestrel'; memory 884810224906779648 is current with the same entities


## 5. What the graph knows now

Three reads that show what ingestion bought: a graph walk from Ada reaches the constraint and the
new maintainer two hops away; provenance on the current maintainer walks back through the
superseded fact to both notes; and an `as_of` read in April answers with the maintainer the
database believed at the time.

In [7]:
print("two hops from Ada:")
for m in db.recall_2hop("Ada", limit=10):
    kind = f" [{m.kind}]" if m.kind != "fact" else ""
    print(f"   {m.valid_from.date()}  {m.content}{kind}")

current = [m for m in db.context("ingest service") if db.provenance(m.memory_id).depth > 0][0]
prov = db.provenance(current.memory_id)
print(f"\nnow: {current.content}")
for link, episode in zip(prov.chain[1:], prov.episodes[1:]):
    print(f"before: {link.content} (valid until {link.valid_to.date()}), from {episode.source}")
print("evidence, newest first:", [e.source for e in prov.episodes])

april = datetime(2026, 4, 1, 12)
print(f"\nas of {april.date()}:", [m.content for m in db.as_of(april).context('ingest service')])
print("today:           ", [m.content for m in db.context("ingest service")])

two hops from Ada:


   2026-08-20  The ingest service must stay on Python 3.10 until the Kestrel migration finishes [constraint]
   2026-06-15  Cy maintains the ingest service
   2026-03-02  Kestrel owns the ingest service
   2026-03-02  Ada leads Kestrel

now: Cy maintains the ingest service
before: Bo maintains the ingest service (valid until 2026-06-15), from notes/2026-03-02.md
evidence, newest first: ['notes/2026-06-15.md', 'notes/2026-03-02.md']



as of 2026-04-01: ['Bo maintains the ingest service', 'Kestrel owns the ingest service']
today:            ['The ingest service must stay on Python 3.10 until the Kestrel migration finishes', 'Cy maintains the ingest service', 'Kestrel owns the ingest service']


## 6. A real extractor

`OpenAICompatibleExtractor` talks to any OpenAI-compatible chat endpoint. It sends the note and the
existing facts about the entities the note names, and parses the JSON patch the model returns; the
pipeline after that is exactly what ran above. The cell runs only when a key is in the environment.

Over MCP the same flow is two tool calls, `ingest` (propose) and `apply_patch` (commit), and in the
OpenAI Agents SDK it is the `anatid_ingest` tool; see
[`04_agents_and_mcp.ipynb`](04_agents_and_mcp.ipynb). The answer-quality benchmark in
[`docs/quality.md`](../../docs/quality.md) measures what a small model's extraction costs against
perfect patches, and is where the pipeline's handover step came from.

In [8]:
import os

key = os.environ.get("OPEN_ROUTER_KEY") or os.environ.get("OPENROUTER_API_KEY")
if key:
    from anatid.ingest import OpenAICompatibleExtractor

    live = OpenAICompatibleExtractor(model="z-ai/glm-5.3-flash", base_url="https://openrouter.ai/api/v1", api_key=key)
    proposal = propose(db, "Dee joined the Kestrel team and will pair with Cy on the ingest service.", extractor=live)
    print(proposal.describe())
else:
    print("Set OPEN_ROUTER_KEY to run a live extraction; the pipeline is otherwise identical.")
db.close()

Set OPEN_ROUTER_KEY to run a live extraction; the pipeline is otherwise identical.
